In [ ]:
from sklearn.metrics import confusion_matrix
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import joblib

from src.video_transcription.points_normalization import normalize_lists
from src.letter_detection.letter_detection_models import AnalysisModel

In [ ]:
test_data = pd.read_csv("datasets/anita_video_dataset.csv")
X_test = test_data.iloc[:, :63].values
y_test = test_data.iloc[:, 63:].values

In [ ]:
nn_model = load_model('models/pawel_anita_love_8.keras')

y_pred = nn_model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

cm_nn = confusion_matrix(y_true, y_pred_classes)
cm_nn_normalized = cm_nn.astype('float') / 271
cm_nn_normalized_rounded = cm_nn_normalized.round(2)
cm_nn

In [ ]:
plt.figure(figsize=(15,7))
klasy = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']
sns.heatmap(cm_nn_normalized_rounded, annot=True, fmt='g', cmap=sns.cubehelix_palette(as_cmap=True), xticklabels=klasy, yticklabels=klasy)
plt.plot()
plt.xlabel('Przewidziane')
plt.ylabel('Rzeczywiste')
plt.title('Macierz Pomyłek')

In [ ]:
dt_model = joblib.load("../models_training/models/decision_tree_augmentation_data.pkl")

y_pred = dt_model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

cm_dt = confusion_matrix(y_true, y_pred_classes)
cm_dt_normalized = cm_dt.astype('float') / 271
cm_dt_normalized_rounded = cm_dt_normalized.round(2)

In [ ]:
plt.figure(figsize=(15,7))
klasy = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']
sns.heatmap(cm_dt_normalized_rounded, annot=True, fmt='g', cmap=sns.cubehelix_palette(as_cmap=True), xticklabels=klasy, yticklabels=klasy)
plt.plot()
plt.xlabel('Przewidziane')
plt.ylabel('Rzeczywiste')
plt.title('Macierz Pomyłek')

In [ ]:
X_train = pd.read_csv("datasets/pawel_video_dataset.csv").iloc[:, :63].values
y_train = pd.read_csv("datasets/pawel_video_dataset.csv").iloc[:, 63:].values

In [ ]:
analysis_model = AnalysisModel()

LETTERS = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K',
           'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U',
           'V', 'W', 'X', 'Y']

cm_analysis = np.zeros((24, 24))
for i in range(len(y_train)):
    xyz_points = X_train[i]
    x_points = xyz_points[:21]
    y_points = xyz_points[21:42]
    z_points = xyz_points[42:]
    x_normalized, y_normalized, is_vertical = normalize_lists(x_points, y_points)
    letter = analysis_model.predict(x_normalized, y_normalized, z_points, is_vertical)
    if letter:
        cm_analysis[LETTERS.index(letter)][np.argmax(y_train[i])] += 1    

cm_analysis_normalized = cm_analysis.astype('float') / 801
cm_analysis_normalized_rounded = cm_analysis_normalized.round(2)
plt.figure(figsize=(15,7))
klasy = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']
sns.heatmap(cm_analysis_normalized_rounded, annot=True, fmt='g', cmap=sns.cubehelix_palette(as_cmap=True), xticklabels=klasy, yticklabels=klasy)
plt.plot()
plt.xlabel('Przewidziane')
plt.ylabel('Rzeczywiste')
plt.title('Macierz Pomyłek')

In [ ]:
result = 0
for i in range(24):
        result += 1 - cm_analysis_normalized_rounded[i][i]
result